# Ch.3 座学：画像処理入門（Pillow × NumPy）
### 講義時間：20分

---

> **講師メモ**：最大の山場は「画像 = 数値の配列」という概念の定着。  
> `print(img_array.astype(int))` で実際に数値を表示して「これが画像の正体です」と見せる。  
> 実務ユースケースを先に見せてモチベーションを上げてから、概念説明に入ると効果的。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from sklearn.datasets import load_digits
import japanize_matplotlib

digits = load_digits()
print(f'手書き数字データ：{digits.images.shape[0]} 枚（各 {digits.images.shape[1]}×{digits.images.shape[2]} ピクセル）')

---
## 【スライド1】なぜ画像処理を学ぶのか

### 画像はデータの一種 = Pythonで扱える

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.axis('off')

usecases = [
    ('製造業', '製品の傷・\n欠陥を自動検出', '#4A90D9'),
    ('医療', 'レントゲン・MRI\n画像の解析', '#5BA85A'),
    ('小売', '商品の棚卸し\n在庫確認', '#E8A838'),
    ('金融', '書類・帳票の\n文字認識（OCR）', '#C0392B'),
    ('農業', '衛星画像から\n生育状況を分析', '#8E44AD'),
]

for i, (sector, detail, color) in enumerate(usecases):
    x = 0.01 + i * 0.198
    box = mpatches.FancyBboxPatch((x, 0.08), 0.175, 0.84,
        boxstyle='round,pad=0.02', facecolor=color, edgecolor='white', lw=2,
        transform=ax.transAxes)
    ax.add_patch(box)
    ax.text(x + 0.0875, 0.82, sector, transform=ax.transAxes,
            ha='center', va='top', fontsize=12, color='white', fontweight='bold')
    ax.text(x + 0.0875, 0.40, detail, transform=ax.transAxes,
            ha='center', va='center', fontsize=9.5, color='#FDFEFE')

ax.set_title('画像処理の実務ユースケース', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

---
## 【スライド2】画像の正体：数値の2次元配列

### 「画像を見る」= 「数値の表を見る」

In [ ]:
# デモ：数字「0」の画像の正体を見せる
img_array = digits.images[0]  # 8×8の配列

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# 左：数値の表として表示
axes[0].axis('off')
axes[0].set_title('数値の配列（8×8）', fontsize=12)
table_data = img_array.astype(int)
tb = axes[0].table(cellText=table_data,
                   loc='center',
                   cellLoc='center')
tb.auto_set_font_size(False)
tb.set_fontsize(10)
tb.scale(1.2, 1.5)

# セルの色：値が大きいほど濃く
for (row, col), cell in tb.get_celld().items():
    if row > 0:
        val = table_data[row-1][col]
        intensity = val / 16.0
        cell.set_facecolor((1 - intensity * 0.7, 1 - intensity * 0.7, 1 - intensity * 0.7))

# 右：画像として表示
axes[1].imshow(img_array, cmap='gray_r', interpolation='nearest')
axes[1].set_title('画像として表示', fontsize=12)
axes[1].set_xticks(range(8))
axes[1].set_yticks(range(8))
axes[1].grid(True, color='white', linewidth=0.5)

plt.suptitle('画像の正体：数値の2次元配列（ラベル: 0）', fontsize=13)
plt.tight_layout()
plt.show()

print('ポイント：数値が大きい（=明るい）部分が白く表示される')
print(f'値の範囲: {int(img_array.min())} 〜 {int(img_array.max())}（0=黒 / 16=白）')

---
## 【スライド3】カラー画像の場合（RGB）

### グレースケールとカラーの違い

In [ ]:
# デモ：RGB = 3層構造の説明図
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))

# カラー画像を numpy で作成（グラデーション）
r = np.zeros((40, 40, 3), dtype=np.uint8)
g = np.zeros((40, 40, 3), dtype=np.uint8)
b = np.zeros((40, 40, 3), dtype=np.uint8)
color_img = np.zeros((40, 40, 3), dtype=np.uint8)

for i in range(40):
    for j in range(40):
        rv = int(255 * i / 39)
        gv = int(255 * j / 39)
        bv = 128
        r[i, j] = [rv, 0, 0]
        g[i, j] = [0, gv, 0]
        b[i, j] = [0, 0, bv]
        color_img[i, j] = [rv, gv, bv]

axes[0].imshow(r)
axes[0].set_title('R チャンネル\n（赤成分）', fontsize=10)
axes[0].axis('off')

axes[1].imshow(g)
axes[1].set_title('G チャンネル\n（緑成分）', fontsize=10)
axes[1].axis('off')

axes[2].imshow(b)
axes[2].set_title('B チャンネル\n（青成分）', fontsize=10)
axes[2].axis('off')

axes[3].imshow(color_img)
axes[3].set_title('R+G+B\n（カラー画像）', fontsize=10)
axes[3].axis('off')

plt.suptitle('カラー画像の構造：RGB 3チャンネルの重ね合わせ', fontsize=12)
plt.tight_layout()
plt.show()

print('グレースケール：1チャンネル（明るさのみ）→ 配列の形は (H, W)')
print('カラー（RGB） ：3チャンネル              → 配列の形は (H, W, 3)')
print()
print('グレースケール変換 = 色情報を捨てて、明るさだけ残す')

---
## 【スライド4】Pillow（PIL）で画像を操作する

### numpy array ↔ PIL Image の変換が基本

In [ ]:
# デモ：PIL の基本操作
# 1. numpy array → PIL Image
img_array_scaled = (digits.images[0] / 16.0 * 255).astype(np.uint8)
img_pil = Image.fromarray(img_array_scaled, mode='L')

# 2. リサイズ（8×8 → 80×80）
img_large = img_pil.resize((80, 80), Image.NEAREST)

# 3. PIL Image → numpy array（元に戻す）
arr_back = np.array(img_large)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

axes[0].imshow(img_pil, cmap='gray_r')
axes[0].set_title('① numpy array\n→ PIL Image\n（8×8）', fontsize=10)
axes[0].axis('off')

axes[1].imshow(img_large, cmap='gray_r')
axes[1].set_title('② PIL Image.resize\n（80×80に拡大）', fontsize=10)
axes[1].axis('off')

axes[2].imshow(arr_back, cmap='gray_r')
axes[2].set_title('③ PIL Image\n→ numpy array\n（分析に使える）', fontsize=10)
axes[2].axis('off')

plt.suptitle('Pillow の基本フロー', fontsize=13)
plt.tight_layout()
plt.show()

print('コードの流れ：')
print('  numpy array → Image.fromarray() → PIL Image')
print('  PIL Image   → img.resize()      → リサイズ後の PIL Image')
print('  PIL Image   → np.array()        → numpy array（元に戻す）')

---
## 【スライド5】フィルタ処理：画像の見え方を変える

In [ ]:
# デモ：各種フィルタの効果
idx_8 = np.where(digits.target == 8)[0][0]
arr_8 = (digits.images[idx_8] / 16.0 * 255).astype(np.uint8)
img_base = Image.fromarray(arr_8, mode='L').resize((80, 80), Image.NEAREST)

filters = [
    ('元画像', img_base),
    ('BLUR\nぼかし', img_base.filter(ImageFilter.BLUR)),
    ('SHARPEN\nシャープ', img_base.filter(ImageFilter.SHARPEN)),
    ('CONTOUR\n輪郭抽出', img_base.filter(ImageFilter.CONTOUR)),
    ('EMBOSS\n浮き彫り', img_base.filter(ImageFilter.EMBOSS)),
]

fig, axes = plt.subplots(1, 5, figsize=(14, 3.5))
for ax, (title, img) in zip(axes, filters):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.suptitle('フィルタ処理の種類と効果（数字「8」）', fontsize=13)
plt.tight_layout()
plt.show()

print('フィルタの仕組み：周辺ピクセルの値を使った計算（畳み込み）で変換する')
print()
print('BLUR    : 周辺の平均 → エッジがなめらかになる')
print('SHARPEN : 中心を強調 → エッジが際立つ')
print('CONTOUR : 差分を抽出 → 境界線（輪郭）だけが残る')

---
## 【スライド6】画像データの統計分析

### 画像 = 数値 なので、統計量も計算できる

In [ ]:
# デモ：数字ごとのピクセル量を可視化（「1」は細い、「8」は太い）
brightness = {}
pixel_count = {}  # 0以上のピクセルの数（= 描かれている量）

for digit in range(10):
    indices = np.where(digits.target == digit)[0]
    brightness[digit] = digits.images[indices].mean()
    pixel_count[digit] = (digits.images[indices] > 0).mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 各数字の代表画像を並べる
for d in range(10):
    idx = np.where(digits.target == d)[0][0]
    arr = (digits.images[idx] / 16.0 * 255).astype(np.uint8)
    img = Image.fromarray(arr, mode='L').resize((20, 20), Image.NEAREST)

# 左：平均ピクセル値（明るさ）
axes[0].bar(brightness.keys(), brightness.values(), color='steelblue', edgecolor='white')
axes[0].set_title('数字ごとの平均ピクセル値（明るさ）')
axes[0].set_xlabel('数字')
axes[0].set_ylabel('平均ピクセル値（0〜16）')
axes[0].set_xticks(range(10))

# 右：ピクセルが描かれている割合
axes[1].bar(pixel_count.keys(), pixel_count.values(), color='coral', edgecolor='white')
axes[1].set_title('数字ごとの「描かれているピクセル」の割合')
axes[1].set_xlabel('数字')
axes[1].set_ylabel('割合（%）')
axes[1].set_xticks(range(10))

plt.suptitle('画像データの統計的分析', fontsize=13)
plt.tight_layout()
plt.show()

print('観察：「1」は細くてピクセルが少ない → 平均が低い')
print('      「0」「8」は面積が多い → 平均が高い')
print()
print('→ ピクセルの統計量 = 機械学習の「特徴量」として使える！')

---
## 【スライド7】まとめ

### Ch.3 で覚えておくこと

| 概念 | ポイント |
|------|----------|
| 画像 = 数値の配列 | グレースケール: (H, W)、カラー: (H, W, 3) |
| numpy → PIL | `Image.fromarray(arr, mode='L')` |
| PIL → numpy | `np.array(img)` |
| リサイズ | `img.resize((幅, 高さ), Image.NEAREST)` |
| フィルタ | `img.filter(ImageFilter.BLUR)` など |
| 統計分析 | ピクセル値の mean / std / max を計算できる |

### 重要な概念

> - **グレースケール変換** = 色情報（RGB）を失い、明るさだけ残す
> - **フィルタ処理** = 周辺ピクセルの値を使った計算（畳み込み）
> - **画像の統計量** = 機械学習の特徴量として使える

### 次は「機械学習モデル構築」へ
> 数値データ・画像データの理解を踏まえて、いよいよモデルを作ります。